In [1]:
import pandas as pd
import numpy as np

In [2]:
        
data = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/2022-12-SGFTFN_TITANITES_unprocessed.xlsx",header=0,index_col=0)
# data = pd.concat([data,data1],axis=0)
oxide = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/oxide_data.xlsx", sheet_name="Sheet1",index_col=0,header=0)
oxlist = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5"]

def wt_to_mol(data, oxide = oxide):
    data[data<=2] = 0
    oxlist1 = oxlist.copy()
    oxide = oxide.T[oxlist1].iloc[0, :].to_numpy()
    data = normalize(data)
    [r, c] = data.shape
    data_f = np.empty((r, c))
    for i in range(0, c):
        data_f[:, i] = data[:, i] / oxide[i]
    data
    data_f = normalize(data_f)
    return(data_f.round(2))


def normalize(data):
    [r, c] = data.shape
    a = data.sum(axis=1).reshape((len(data), 1))
    data_formatted = ((data*100)/a).round(1)
    return(data_formatted)


def cat_calc(data1,oxide_list):
    total = data1.columns.get_loc("Total")
    o_no = data1.loc[:,"Oxygen_no"]
    data = data1.iloc[:,:total].copy()
    oxide = oxide_list.loc[data.columns]
    data = data.div(oxide['Mol. Wt.'].values,axis=1).round(3)
    data = data.mul(oxide['O_no'].values,axis=1).round(3)
    norm = o_no.div(data.sum(axis=1)).round(3)
    data = data.mul(norm.values,axis=0).round(3)
    data = data.mul(oxide['Cat_per_o'].values,axis=1).round(3)
    total = data.sum(axis=1).round(3)
    data.columns = oxide['Cation'].values
    data['Cation_Total'] = total
    return data.round(3)


In [3]:
data2 = data[['SIO2(WT%)','TIO2(WT%)','AL2O3(WT%)','CR2O3(WT%)','FE2O3T(WT%)', 'FE2O3(WT%)', 'FEOT(WT%)','FEO(WT%)', 'MNO(WT%)','MGO(WT%)','CAO(WT%)', 'NA2O(WT%)','K2O(WT%)','P2O5(WT%)','MINERAL']]

In [4]:
data2.MINERAL.unique()

array(['TITANITE (SPHENE)', nan], dtype=object)

In [5]:
len(data2)

10038

In [6]:
data_cleaned = data2.loc[(~data2['SIO2(WT%)'].isna()) & (~data2['TIO2(WT%)'].isna()) & (~data2['CAO(WT%)'].isna()),:]
# data_px = data_cleaned.loc[(data_cleaned['MINERAL']=="ILMENITE"),:]
data_px = data_cleaned.copy()
data_px.loc[~data_px['FEOT(WT%)'].isna(),:]
data_px.pop("FE2O3(WT%)")
data_px.pop("FE2O3T(WT%)")
data_px.pop("FEO(WT%)")
data_px.columns = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'Mineral']
mineral = data_px['Mineral']
data_px

,SiO2,TiO2,Al2O3,Cr2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,,
[743] FRIKH-KHAR D. I. (1988),30.52,30.05,0.17,NaN,NaN,0.04,NaN,25.7,0.24,NaN,NaN,TITANITE (SPHENE)
[743] FRIKH-KHAR D. I. (1988),30.35,34.93,1.87,NaN,NaN,0.13,NaN,25.25,NaN,NaN,NaN,TITANITE (SPHENE)
[743] FRIKH-KHAR D. I. (1988),29.18,35.51,1.40,NaN,NaN,0.18,NaN,25.28,NaN,NaN,NaN,TITANITE (SPHENE)
[743] FRIKH-KHAR D. I. (1988),29.88,36.16,1.53,NaN,NaN,0.13,NaN,25.39,NaN,NaN,NaN,TITANITE (SPHENE)
[766] DIXON T. H. (1984),30.53,36.38,1.22,NaN,0.83,0.19,0.04,28.81,0.01,NaN,NaN,TITANITE (SPHENE)
...,...,...,...,...,...,...,...,...,...,...,...,...
[26118] ELIZONDO-PACHECO L. A. (2022),29.8,35.13,2.21,0.00,1.36,0.07,0.00,28.87,0.00,0.01,0.12,TITANITE (SPHENE)
[26118] ELIZONDO-PACHECO L. A. (2022),29.48,37.2,1.55,0.00,1.00,0.04,0.00,29.65,0.11,0.00,0.12,TITANITE (SPHENE)
[26118] ELIZONDO-PACHECO L. A. (2022),29.91,37.49,1.83,0.01,1.24,0.06,0.00,29.83,0.07,0.01,0.06,TITANITE (SPHENE)


In [7]:
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')
data_px2 = data_px.fillna(0)
total = data_px2.sum(axis=1)
data_px2['Mineral'] = mineral
data_px2 = data_px2[(total>97) & (total<101)]
mineral = data_px2.pop("Mineral")
col = data_px2.columns
ind = data_px2.index
data_px2 = pd.DataFrame(wt_to_mol(data_px2.to_numpy(),oxide),columns = col,index=ind)
# data_px2 = data
data_px2

,SiO2,TiO2,Al2O3,Cr2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5
CITATION,,,,,,,,,,,
[766] DIXON T. H. (1984),34.4,30.8,0.0,0.0,0.0,0.0,0.0,34.8,0.0,0.0,0.0
[766] DIXON T. H. (1984),33.4,31.7,0.0,0.0,0.0,0.0,0.0,34.9,0.0,0.0,0.0
[1929] HELLMAN P. L. (1985),34.8,25.8,1.9,0.0,2.4,0.0,0.0,35.0,0.0,0.0,0.0
[1929] HELLMAN P. L. (1985),36.2,24.6,2.6,0.0,3.0,0.0,0.0,33.6,0.0,0.0,0.0
[2670] KAY S. M. (1983),34.3,29.3,1.6,0.0,0.0,0.0,0.0,34.8,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
[26118] ELIZONDO-PACHECO L. A. (2022),33.6,29.9,1.5,0.0,0.0,0.0,0.0,35.0,0.0,0.0,0.0
[26118] ELIZONDO-PACHECO L. A. (2022),33.0,31.3,0.0,0.0,0.0,0.0,0.0,35.6,0.0,0.0,0.0
[26118] ELIZONDO-PACHECO L. A. (2022),33.2,31.3,0.0,0.0,0.0,0.0,0.0,35.5,0.0,0.0,0.0


In [8]:
non_essential_sum = data_px2[["FeO","MnO","MgO","Na2O", "K2O"]].sum(axis=1)
data_px2 = data_px2[non_essential_sum<3]
# m = data_px2[["FeO","MnO","MgO"]].sum(axis=1)
# data_px2 = data_px2[ (m >= 49) & (m <= 51)]
# data_px2 = data_px2[(data_px2.Al2O3 + data_px2.Cr2O3 >= 49) & (data_px2.Al2O3 + data_px2.Cr2O3 <= 51)]
# data_px2['P2O5'] = 0
data_px2.loc[:,'Mineral'] = "Ttn"
data_px2.loc[:,'Al2O3'] = data_px2['Al2O3'] + data_px2['Cr2O3']
data_px2.pop("Cr2O3")
data_px2
data_px2.to_excel("/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/molar tables/new data/Ttn_processed_mol.xlsx")

/var/folders/qb/qlnn195d5kqdd5s0frpdjndm0000gn/T/ipykernel_1468/1349351576.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_px2.loc[:,'Mineral'] = "Ttn"


In [9]:
data_px2.sort_values("TiO2",ascending=False)

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,
[24384] SENSARMA S. (2021),20.5,60.6,0.0,0.0,0.0,0.0,18.8,0.0,0.0,0.0,Ttn
[21058] SHAIKH A. M. (2017),28.9,35.6,0.0,0.0,0.0,0.0,35.5,0.0,0.0,0.0,Ttn
[23218] PEREZ-SOBA C. (2019),31.6,35.4,0.0,0.0,0.0,0.0,33.0,0.0,0.0,0.0,Ttn
[23218] PEREZ-SOBA C. (2019),31.4,34.4,0.0,0.0,0.0,0.0,34.2,0.0,0.0,0.0,Ttn
[19308] KRMICEK L. (2011),33.9,34.2,0.0,0.0,0.0,0.0,31.9,0.0,0.0,0.0,Ttn
...,...,...,...,...,...,...,...,...,...,...,...
[21504] CAO MINGJIAN (2017),36.4,22.3,6.1,0.0,0.0,0.0,35.2,0.0,0.0,0.0,Ttn
[21504] CAO MINGJIAN (2017),36.4,22.2,6.2,0.0,0.0,0.0,35.1,0.0,0.0,0.0,Ttn
[24177] GROS K. (2020),36.8,21.7,6.7,0.0,0.0,0.0,34.8,0.0,0.0,0.0,Ttn


In [10]:
data_cleaned = data2.loc[~data2['FEO(WT%)'].isna(),:]
# data_px = data_cleaned.loc[(data_cleaned['MINERAL']=="ILMENITE"),:]
data_px = data_cleaned.copy()
data_px.loc[~data_px['FEOT(WT%)'].isna(),:]
data_px.pop("FE2O3(WT%)")
data_px.pop("FE2O3T(WT%)")
data_px.pop("FEO(WT%)")
data_px.columns = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'Mineral']
mineral = data_px['Mineral']
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')
data_px2 = data_px.fillna(0)
total = data_px2.sum(axis=1)
data_px2['Mineral'] = mineral
data_px2 = data_px2[(total>98) & (total<101)]
mineral = data_px2.pop("Mineral")
col = data_px2.columns
ind = data_px2.index
data_px2 = pd.DataFrame(wt_to_mol(data_px2.to_numpy(),oxide),columns = col,index=ind)
# data_px2 = data
data_px2

non_essential_sum = data_px2[['SiO2',"Al2O3","MnO","MgO","CaO", "Na2O", "K2O"]].sum(axis=1)
data_px2 = data_px2[non_essential_sum<10]
m = data_px2[["FeO","MnO","MgO"]].sum(axis=1)
data_px2 = data_px2[ (m >= 90) & (m <= 100)]
# data_px2 = data_px2[(data_px2.Al2O3 + data_px2.Cr2O3 >= 49) & (data_px2.Al2O3 + data_px2.Cr2O3 <= 51)]
# data_px2['P2O5'] = 0
data_px2['Mineral'] = "Mag/Hem"
data_px2['Al2O3'] = data_px2['Al2O3'] + data_px2['Cr2O3']
data_px2.pop("Cr2O3")
data_px2

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,


In [11]:
data_cleaned[data_cleaned["MINERAL"]=="CHROME-SPINEL"]

,SIO2(WT%),TIO2(WT%),AL2O3(WT%),CR2O3(WT%),FE2O3T(WT%),FE2O3(WT%),FEOT(WT%),FEO(WT%),MNO(WT%),MGO(WT%),CAO(WT%),NA2O(WT%),K2O(WT%),P2O5(WT%),MINERAL
CITATION,,,,,,,,,,,,,,,
